# Phase 6 — Deep learning (PyTorch + TensorFlow)

Same leakage-safe features as the GBM, trained with a neural net.

In [1]:
import sys; sys.path.append('..')
from src.seed import seed_everything; seed_everything(42)


## 6a — PyTorch (end-to-end, with early stopping on a temporal val split)

In [2]:
from src.train_torch import main as torch_main
torch_result = torch_main()   # prints test PR-AUC + 95% CI


[PyTorch MLP] test PR-AUC = 0.160  (95% CI [0.105, 0.233])  best val PR-AUC=0.115


## 6b — TensorFlow/Keras

`src/models_tf.py` builds the equivalent Keras model with PR-AUC metric, EarlyStopping, and `class_weight` for imbalance.

In [3]:
from src.models_tf import build_model, early_stopping
from src.data import get_data
from src.train import engineer, temporal_split, FEATURES
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score
import numpy as np

df = engineer(get_data()[0])
tr, te = temporal_split(df)
imp = SimpleImputer(strategy='median').fit(tr[FEATURES])
sc = StandardScaler().fit(imp.transform(tr[FEATURES]))   # TRAIN-ONLY fit
Xtr = sc.transform(imp.transform(tr[FEATURES])); Xte = sc.transform(imp.transform(te[FEATURES]))
ytr = tr.is_fraud.to_numpy(); yte = te.is_fraud.to_numpy()
pw = (len(ytr)-ytr.sum())/ytr.sum()
m = build_model(len(FEATURES))
m.fit(Xtr, ytr, validation_split=0.15, epochs=40, verbose=0,
      class_weight={0:1.0, 1:float(pw)}, callbacks=[early_stopping()])
print('Keras test PR-AUC:', round(average_precision_score(yte, m.predict(Xte, verbose=0).ravel()),3))


Keras test PR-AUC: 0.124
